# 📄🤝 CVMatch AI — CV and job ad matching with Gemini

**Project:** `cvmatch-ai`
**Task:** Deep Learning, assignment 1, part 1.2.2 — a working prototype of a generative AI system in Google Colab using the **Gemini API**.

**Problem:** job seekers send the same CV to dozens of job ads and don't know *why* they get no answer; recruiters read hundreds of CVs a day.
**CVMatch AI** takes a candidate's **CV** and a **job advertisement** (input `X`) and returns a structured evaluation (output `y`):

```json
{
  "match_score": 85,
  "verdict": "Strong match",
  "matched_skills": ["Python (3+ years)", "Django", "SQL / PostgreSQL"],
  "missing_skills": ["AWS", "Kubernetes"],
  "recommendations": ["If you have any AWS experience, add it to the CV"]
}
```

Each numbered section of this notebook is one UML component from the README:

| Section | UML component | What it does |
| --- | --- | --- |
| 3 | GitHub Repository → Prompt Template | Load `prompts/prompt.json` and `data/examples.json` |
| 4 | Input Processor | Validate and clean the CV and job ad |
| 5 | Prompt Builder | Fill the prompt template |
| 6 | Gemini API | `generate_content` with a JSON response schema (+ offline fallback) |
| 7 | Result Validator | Check the JSON, clamp the score, match the verdict to the score |
| 8 | REST API `POST /match` | `post_match(cv, job_ad)` connects all components |
| 9 | End-to-end | Run all `(X, y)` examples, compare with `y`, visualise |

**How to run:** add a Colab Secret named `GEMINI_API_KEY` (🔑 panel on the left, key from [Google AI Studio](https://aistudio.google.com/apikey), enable *Notebook access*), then **Runtime → Run all**.
Without a key the notebook still runs end-to-end: the Gemini step is skipped and a simple keyword-overlap fallback returns the same JSON shape.

## 1. System overview

The user sends a CV and a job ad to `POST /match`. The **Input Processor** cleans the texts, the **Prompt Builder** fills the prompt template from GitHub, **Gemini** evaluates the match and returns JSON, and the **Result Validator** checks that JSON before it is returned to the user.

### Component diagram

```mermaid
flowchart LR

    User["Job Seeker / Recruiter"]

    subgraph CVMatch["CVMatch AI"]
        API["REST API<br/>POST /match"]
        Input["Input Processor"]
        Prompt["Prompt Builder"]
        Gemini["Gemini API"]
        Validator["Result Validator"]
    end

    Template[/"Prompt Template<br/>(prompt.json)"/]
    GitHub["GitHub Repository"]

    User -->|"CV text +<br/>job ad text"| API

    API -->|"Validate & clean<br/>input"| Input

    Input -->|"Clean CV +<br/>job ad"| Prompt

    Template -->|"System prompt +<br/>user template"| Prompt

    Prompt -->|"Filled prompt +<br/>JSON schema"| Gemini

    Gemini -->|"Score + skills +<br/>recommendations + verdict"| Validator

    Validator -->|"Validated result"| API

    API -->|"JSON response"| User

    GitHub -->|"Prompt file"| Template
```

### Sequence diagram

```mermaid
sequenceDiagram
    actor User as Job Seeker / Recruiter
    participant API as REST API /match
    participant Input as Input Processor
    participant Prompt as Prompt Builder
    participant Gemini as Gemini API
    participant Validator as Result Validator

    User->>API: POST /match<br/>{cv, job_ad}
    API->>Input: Validate & clean texts
    Input-->>API: Clean CV + job ad + warnings

    API->>Prompt: CV + job ad
    Prompt->>Prompt: Fill prompt.json template
    Prompt-->>API: System prompt + user prompt

    API->>Gemini: generate_content(prompt, response_schema)
    alt Gemini answered
        Gemini-->>Validator: JSON answer
    else No API key / API error
        API->>Validator: Keyword-overlap fallback result
    end

    Validator->>Validator: Check fields, clamp score,<br/>match verdict to score
    Validator-->>API: Validated result

    API-->>User: JSON response<br/>{match_score, verdict, matched_skills,<br/>missing_skills, recommendations}
```

In [ ]:
#@title 🖼️ Render the UML diagrams (Colab does not render Mermaid in Markdown)
import base64
from IPython.display import Image, display

COMPONENT_MERMAID = """flowchart LR

    User["Job Seeker / Recruiter"]

    subgraph CVMatch["CVMatch AI"]
        API["REST API<br/>POST /match"]
        Input["Input Processor"]
        Prompt["Prompt Builder"]
        Gemini["Gemini API"]
        Validator["Result Validator"]
    end

    Template[/"Prompt Template<br/>(prompt.json)"/]
    GitHub["GitHub Repository"]

    User -->|"CV text +<br/>job ad text"| API

    API -->|"Validate & clean<br/>input"| Input

    Input -->|"Clean CV +<br/>job ad"| Prompt

    Template -->|"System prompt +<br/>user template"| Prompt

    Prompt -->|"Filled prompt +<br/>JSON schema"| Gemini

    Gemini -->|"Score + skills +<br/>recommendations + verdict"| Validator

    Validator -->|"Validated result"| API

    API -->|"JSON response"| User

    GitHub -->|"Prompt file"| Template"""

SEQUENCE_MERMAID = """sequenceDiagram
    actor User as Job Seeker / Recruiter
    participant API as REST API /match
    participant Input as Input Processor
    participant Prompt as Prompt Builder
    participant Gemini as Gemini API
    participant Validator as Result Validator

    User->>API: POST /match<br/>{cv, job_ad}
    API->>Input: Validate & clean texts
    Input-->>API: Clean CV + job ad + warnings

    API->>Prompt: CV + job ad
    Prompt->>Prompt: Fill prompt.json template
    Prompt-->>API: System prompt + user prompt

    API->>Gemini: generate_content(prompt, response_schema)
    alt Gemini answered
        Gemini-->>Validator: JSON answer
    else No API key / API error
        API->>Validator: Keyword-overlap fallback result
    end

    Validator->>Validator: Check fields, clamp score,<br/>match verdict to score
    Validator-->>API: Validated result

    API-->>User: JSON response<br/>{match_score, verdict, matched_skills,<br/>missing_skills, recommendations}"""

for diagram in (COMPONENT_MERMAID, SEQUENCE_MERMAID):
    encoded = base64.urlsafe_b64encode(diagram.encode("utf-8")).decode("ascii")
    display(Image(url=f"https://mermaid.ink/img/{encoded}?type=png&bgColor=white"))

## 2. Setup — packages and `GEMINI_API_KEY`

We use the official **Google Gen AI SDK** (`google-genai`). The API key is read from **Colab Secrets**, the same way as in the course example (`userdata.get(...)`), and is never printed.
If the key is missing, `HAS_GEMINI = False` and the Gemini step is skipped instead of crashing.

In [ ]:
#@title 🔧 2.1 Install packages
%pip install -q -U google-genai pandas matplotlib

In [ ]:
#@title 🔑 2.2 Read the API key and create the Gemini client
import os

MODEL = "gemini-flash-latest"  #@param {type:"string"}

GEMINI_API_KEY = None
try:
    from google.colab import userdata

    try:
        GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
    except Exception as exc:  # SecretNotFoundError / NotebookAccessError
        print(f"Colab Secret GEMINI_API_KEY is not available ({type(exc).__name__}).")
except ImportError:  # running outside Colab
    GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")

HAS_GEMINI = bool(GEMINI_API_KEY)
client = None
if HAS_GEMINI:
    from google import genai

    client = genai.Client(api_key=GEMINI_API_KEY)
    print(f"✅ Gemini client ready, model: {MODEL}")
else:
    print("⚠️ No GEMINI_API_KEY — the Gemini step will be skipped and the keyword fallback used.")

## 3. Load data from GitHub (UML: GitHub Repository → Prompt Template)

| File | Content |
| --- | --- |
| `prompts/prompt.json` | `system_prompt`, `user_prompt_template` with `{{cv}}` and `{{job_ad}}`, output format |
| `data/examples.json` | `(X, y)` pairs: `X = {cv, job_ad}`, `y` = expected JSON answer |

If the notebook runs inside a local copy of the repository, the local files are used; otherwise (Colab) they are downloaded from GitHub.

In [ ]:
#@title 📥 3.1 Load prompt and examples
import json
import urllib.request
from pathlib import Path

GITHUB_REPO = "jauniusc/cvmatch"  #@param {type:"string"}
RAW_BASE = f"https://raw.githubusercontent.com/{GITHUB_REPO}/main"


def load_repo_json(path: str):
    for local in (Path(path), Path("..") / path):
        if local.is_file():
            print(f"Loaded {path} (local copy)")
            return json.loads(local.read_text(encoding="utf-8"))
    url = f"{RAW_BASE}/{path}"
    with urllib.request.urlopen(url, timeout=30) as response:
        print(f"Loaded {path} from {url}")
        return json.load(response)


prompt_spec = load_repo_json("prompts/prompt.json")
examples = load_repo_json("data/examples.json")
print(f"\n{len(examples)} examples, prompt version {prompt_spec['version']}")

In [ ]:
#@title 📊 3.2 Data overview
import pandas as pd

pd.set_option("display.max_colwidth", 80)
display(pd.DataFrame([
    {
        "id": ex["id"],
        "scenario": ex["scenario"],
        "CV chars": len(ex["X"]["cv"]),
        "job ad chars": len(ex["X"]["job_ad"]),
        "y.match_score": ex["y"]["match_score"],
        "y.verdict": ex["y"]["verdict"],
    }
    for ex in examples
]))

print("example_01 — X.job_ad:\n")
print(examples[0]["X"]["job_ad"])

## 4. Input Processor

The first component after `POST /match`:
1. **validates** the input — an empty CV or job ad is an error;
2. **cleans** the text — removes extra spaces and blank lines;
3. **limits** the length (very long texts are cut) and **warns** when the CV is very short.

In [ ]:
#@title 🧹 4.1 Validate and clean the input
import re

MIN_CV_CHARS = 200
MAX_CHARS = 15_000


def clean_text(text: str) -> str:
    lines = [re.sub(r"\s+", " ", line).strip() for line in text.strip().splitlines()]
    return re.sub(r"\n{3,}", "\n\n", "\n".join(lines))


def process_input(cv: str, job_ad: str) -> dict:
    if not (cv or "").strip() or not (job_ad or "").strip():
        raise ValueError("Both cv and job_ad must be non-empty text.")
    cv, job_ad = clean_text(cv)[:MAX_CHARS], clean_text(job_ad)[:MAX_CHARS]
    warnings = []
    if len(cv) < MIN_CV_CHARS:
        warnings.append(f"CV is very short ({len(cv)} characters), the evaluation may be unreliable.")
    return {"cv": cv, "job_ad": job_ad, "warnings": warnings}


for ex in examples:
    processed = process_input(ex["X"]["cv"], ex["X"]["job_ad"])
    print(f"{ex['id']}: CV {len(processed['cv'])} chars, warnings: {processed['warnings'] or '-'}")

## 5. Prompt Builder

The prompt is stored in the repository, not in the code (`prompts/prompt.json`):
- `system_prompt` — the role (experienced recruiter), scoring rules (70% required / 30% nice-to-have, never invent skills, verdict thresholds);
- `user_prompt_template` — the CV, the job ad and the JSON structure to return.

The same prompt can also be given to ChatGPT with any example from `data/examples.json` to test it by hand (part 1.2.1).

In [ ]:
#@title 🧾 5.1 Fill the prompt template
SYSTEM_PROMPT = prompt_spec["system_prompt"]


def build_prompt(cv: str, job_ad: str) -> str:
    return prompt_spec["user_prompt_template"].replace("{{cv}}", cv).replace("{{job_ad}}", job_ad)


print("SYSTEM PROMPT\n" + "-" * 60)
print(SYSTEM_PROMPT)
print("\nUSER PROMPT for example_01\n" + "-" * 60)
print(build_prompt(examples[0]["X"]["cv"], examples[0]["X"]["job_ad"]))

## 6. Gemini API

`client.models.generate_content` is called with:
- `system_instruction` — the system prompt;
- `response_mime_type="application/json"` and `response_schema` (a Pydantic model) — Gemini **must** return JSON with the `y` fields;
- `temperature=0.2` — more stable, less "creative" scores.

If the request fails (for example `429` rate limit on the free tier) it is retried. Without an API key, a **keyword-overlap fallback** is used instead: it finds known skills in the job ad and checks which of them appear in the CV. It is much less accurate than Gemini, but it returns the same JSON shape, so the rest of the pipeline still works.

In [ ]:
#@title ✨ 6.1 Gemini call
import time

from pydantic import BaseModel


class MatchResult(BaseModel):
    match_score: int
    verdict: str
    matched_skills: list[str]
    missing_skills: list[str]
    recommendations: list[str]


def call_gemini(user_prompt: str, retries: int = 3) -> str | None:
    """Return Gemini's raw JSON text, or None if there is no key or every attempt failed."""
    if not HAS_GEMINI:
        return None
    from google.genai import types

    config = types.GenerateContentConfig(
        system_instruction=SYSTEM_PROMPT,
        temperature=prompt_spec["generation_config"]["temperature"],
        response_mime_type="application/json",
        response_schema=MatchResult,
    )
    for attempt in range(1, retries + 1):
        try:
            response = client.models.generate_content(model=MODEL, contents=user_prompt, config=config)
            return response.text
        except Exception as exc:
            print(f"Gemini attempt {attempt}/{retries} failed: {type(exc).__name__}: {str(exc)[:150]}")
            if attempt < retries:
                time.sleep(10 * attempt)
    return None


raw = call_gemini(build_prompt(examples[0]["X"]["cv"], examples[0]["X"]["job_ad"]))
print(raw if raw else "Gemini skipped (no API key or request failed).")

In [ ]:
#@title 🔁 6.2 Offline fallback: keyword overlap
SKILL_KEYWORDS = {
    "Python": ["Python"],
    "Java": ["Java"],
    "JavaScript": ["JavaScript"],
    "TypeScript": ["TypeScript"],
    "React": ["React"],
    "Node.js": ["Node.js"],
    "Django": ["Django"],
    "FastAPI": ["FastAPI"],
    "Spring Boot": ["Spring Boot"],
    "SQL": ["SQL", "PostgreSQL", "MySQL", "relational database"],
    "REST API": ["REST API", "REST APIs"],
    "Docker": ["Docker"],
    "Kubernetes": ["Kubernetes"],
    "Terraform": ["Terraform"],
    "AWS": ["AWS"],
    "Azure": ["Azure"],
    "Linux": ["Linux"],
    "Bash": ["Bash"],
    "CI/CD": ["CI/CD", "GitLab CI", "GitHub Actions", "Jenkins"],
    "Celery": ["Celery"],
    "Test automation": ["test automation", "JUnit", "pytest", "Selenium"],
    "Testing theory": ["testing theory", "test cases", "ISTQB"],
    "Jira": ["Jira"],
    "Agile": ["Agile", "Scrum", "Kanban"],
    "English": ["English"],
}

VERDICTS = prompt_spec["output_format"]["verdicts"]  # {"Strong match": [80, 100], ...}


def verdict_for(score: int) -> str:
    return next(name for name, (low, high) in VERDICTS.items() if low <= score <= high)


def find_skills(text: str) -> list[str]:
    def mentioned(word):
        return re.search(rf"(?<!\w){re.escape(word)}(?!\w)", text, re.IGNORECASE)

    return [skill for skill, words in SKILL_KEYWORDS.items() if any(mentioned(w) for w in words)]


def keyword_fallback(cv: str, job_ad: str) -> dict:
    wanted = find_skills(job_ad)
    have = set(find_skills(cv))
    matched = [s for s in wanted if s in have]
    missing = [s for s in wanted if s not in have]
    score = round(100 * len(matched) / len(wanted)) if wanted else 0
    return {
        "match_score": score,
        "verdict": verdict_for(score),
        "matched_skills": matched,
        "missing_skills": missing,
        "recommendations": [f"If you have experience with {s}, mention it in the CV." for s in missing[:3]],
    }


print(json.dumps(keyword_fallback(examples[0]["X"]["cv"], examples[0]["X"]["job_ad"]), indent=2))

## 7. Result Validator

We do not trust the model's answer blindly. The validator:
1. **parses the JSON** (removes ```` ``` ```` fences if the model added them);
2. **checks that all required fields** from `prompt.json` are present;
3. **clamps** `match_score` to 0–100;
4. **makes sure the verdict matches the score** (e.g. 85 must be "Strong match") and corrects it if not.

In [ ]:
#@title ✅ 7.1 Validate the result
REQUIRED_FIELDS = prompt_spec["output_format"]["required_fields"]


def validate_result(raw: str | dict) -> tuple[dict, list[str]]:
    """Return (result in the y shape, warnings). Raises ValueError if the answer is unusable."""
    if isinstance(raw, str):
        raw = json.loads(re.sub(r"^```(?:json)?\s*|\s*```$", "", raw.strip()))
    missing = [f for f in REQUIRED_FIELDS if f not in raw]
    if missing:
        raise ValueError(f"missing fields: {missing}")

    warnings = []
    score = max(0, min(100, int(raw["match_score"])))
    verdict = verdict_for(score)
    if raw["verdict"] != verdict:
        warnings.append(f"verdict '{raw['verdict']}' did not match score {score}, corrected to '{verdict}'")
    result = {"match_score": score, "verdict": verdict}
    for field in ("matched_skills", "missing_skills", "recommendations"):
        result[field] = [str(item) for item in raw[field]]
    return result, warnings


# Demo: an answer with a score above 100 and a verdict that does not match the score
bad_answer = '```json\n{"match_score": 120, "verdict": "Weak match", "matched_skills": ["Python"], "missing_skills": [], "recommendations": []}\n```'
fixed, fixes = validate_result(bad_answer)
print(fixed)
print(fixes)

## 8. REST API `POST /match`

`post_match()` connects all components in the order of the sequence diagram and is the function that a real `POST /match` endpoint would call:

```
POST /match   {"cv": "...", "job_ad": "..."}   →   {"result": {...y...}, "meta": {...}}
```

`meta` shows where the answer came from (`gemini` or `keyword_fallback`), the model, the time taken and any warnings.

In [ ]:
#@title 🌐 8.1 post_match(cv, job_ad)
def post_match(cv: str, job_ad: str) -> dict:
    start = time.time()
    processed = process_input(cv, job_ad)                          # Input Processor
    raw = call_gemini(build_prompt(processed["cv"], processed["job_ad"]))  # Prompt Builder + Gemini API
    warnings = list(processed["warnings"])
    source = "gemini"
    if raw is not None:
        try:
            result, fixes = validate_result(raw)                   # Result Validator
            warnings += fixes
        except (ValueError, TypeError, KeyError) as exc:
            warnings.append(f"Invalid Gemini answer ({exc}), using keyword fallback.")
            raw = None
    if raw is None:
        source = "keyword_fallback"
        result, fixes = validate_result(keyword_fallback(processed["cv"], processed["job_ad"]))
        warnings += fixes
    meta = {
        "source": source,
        "model": MODEL if source == "gemini" else None,
        "seconds": round(time.time() - start, 1),
        "warnings": warnings,
    }
    return {"result": result, "meta": meta}


response = post_match(examples[0]["X"]["cv"], examples[0]["X"]["job_ad"])
print(json.dumps(response, indent=2, ensure_ascii=False))

## 9. End-to-end: all `(X, y)` examples

For each example in `data/examples.json` we call `post_match(X.cv, X.job_ad)` and compare the answer with the expected `y`.

**Metrics:**
- **Mean absolute error (MAE)** of `match_score` — a difference of ±10 points is acceptable, because CV screening is partly subjective;
- **Verdict accuracy** — how often the `verdict` equals `y.verdict`.

In [ ]:
#@title 🚀 9.1 Run all examples
runs = []
for ex in examples:
    print(f"▶ {ex['id']}: {ex['scenario']}")
    runs.append({"example": ex, "response": post_match(ex["X"]["cv"], ex["X"]["job_ad"])})
    if HAS_GEMINI:
        time.sleep(4)  # stay under the free-tier requests-per-minute limit

results_df = pd.DataFrame([
    {
        "id": run["example"]["id"],
        "expected score": run["example"]["y"]["match_score"],
        "predicted score": run["response"]["result"]["match_score"],
        "abs error": abs(run["example"]["y"]["match_score"] - run["response"]["result"]["match_score"]),
        "expected verdict": run["example"]["y"]["verdict"],
        "predicted verdict": run["response"]["result"]["verdict"],
        "source": run["response"]["meta"]["source"],
    }
    for run in runs
])
display(results_df)

mae = results_df["abs error"].mean()
verdict_accuracy = (results_df["expected verdict"] == results_df["predicted verdict"]).mean()
print(f"MAE: {mae:.1f} points   |   verdict accuracy: {verdict_accuracy:.0%}")

In [ ]:
#@title 📈 9.2 Expected vs predicted score
import matplotlib.pyplot as plt
import numpy as np

EXPECTED_COLOR, PREDICTED_COLOR = "#2a78d6", "#eb6834"
source_label = f"Gemini ({MODEL})" if HAS_GEMINI else "keyword fallback (no API key)"

x = np.arange(len(results_df))
width = 0.36
fig, ax = plt.subplots(figsize=(9, 4.5))

for i, (name, (low, high)) in enumerate(VERDICTS.items()):  # verdict bands in the background
    ax.axhspan(low, high + 1, color="#f1f0ec" if i % 2 == 0 else "white", zorder=0)
    ax.text(len(results_df) - 0.45, (low + high) / 2, name, va="center", fontsize=8, color="#52514e")

bars_y = ax.bar(x - width / 2 - 0.01, results_df["expected score"], width, color=EXPECTED_COLOR, label="expected (y)", zorder=2)
bars_p = ax.bar(x + width / 2 + 0.01, results_df["predicted score"], width, color=PREDICTED_COLOR, label=f"predicted: {source_label}", zorder=2)
ax.bar_label(bars_y, padding=2, fontsize=9, color="#0b0b0b")
ax.bar_label(bars_p, padding=2, fontsize=9, color="#0b0b0b")

ax.set_xticks(x, results_df["id"])
ax.set_xlim(-0.6, len(results_df) + 0.4)
ax.set_ylim(0, 105)
ax.set_ylabel("match_score")
ax.set_title(f"Expected vs predicted match score — MAE {mae:.1f}, verdict accuracy {verdict_accuracy:.0%}")
ax.spines[["top", "right"]].set_visible(False)
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.1), ncol=2, frameon=False)
plt.tight_layout()
plt.show()

In [ ]:
#@title 🪪 9.3 Report for each example
from html import escape

from IPython.display import HTML


def chips(items, background):
    if not items:
        return "<i>none</i>"
    return " ".join(
        f'<span style="background:{background};padding:2px 8px;border-radius:10px;margin:2px;display:inline-block">{escape(s)}</span>'
        for s in items
    )


def show_report(example: dict, response: dict):
    r, m, y = response["result"], response["meta"], example["y"]
    recommendations = "".join(f"<li>{escape(s)}</li>" for s in r["recommendations"])
    warnings = f"<p style='color:#9a6700'>⚠️ {escape('; '.join(m['warnings']))}</p>" if m["warnings"] else ""
    display(HTML(f"""
    <div style="border:1px solid #ddd;border-radius:8px;padding:12px 16px;margin:10px 0;font-family:sans-serif;color:#222;background:#fff">
      <h4 style="margin:0 0 6px">{example['id']} — {escape(example['scenario'])}</h4>
      <p style="margin:4px 0"><b>Score:</b> {r['match_score']} / 100 — <b>{r['verdict']}</b>
         &nbsp; <span style="color:#666">(expected {y['match_score']} — {y['verdict']}; source: {m['source']})</span></p>
      <p style="margin:4px 0"><b>✓ Matched:</b> {chips(r['matched_skills'], '#d7f3e6')}</p>
      <p style="margin:4px 0"><b>✗ Missing:</b> {chips(r['missing_skills'], '#fde0df')}</p>
      <b>Recommendations:</b><ul style="margin:4px 0">{recommendations}</ul>{warnings}
    </div>"""))


for run in runs:
    show_report(run["example"], run["response"])

## 10. Try it with your own CV

Paste your own CV and a job ad into the variables below and run the cell. The default is a new case that is not in `data/` (frontend developer → React Native role).

In [ ]:
#@title ✍️ 10.1 Your CV and job ad
MY_CV = """
Laura Mikalauskaitė
Frontend Developer

EXPERIENCE
2022–present: Frontend Developer, Travel Platform Ltd, Vilnius. Building user interfaces with React and TypeScript,
writing component tests with Jest, working with designers in Figma. The team works in Scrum.
2021–2022: Junior Web Developer, Website Studio. HTML, CSS, JavaScript, WordPress themes.

EDUCATION
Vilnius Gediminas Technical University, BSc in Informatics Engineering (2021).

SKILLS
JavaScript, TypeScript, React, Redux, HTML, CSS, REST API, Git, Jest.

LANGUAGES
Lithuanian – native, English – B2.
"""

MY_JOB_AD = """
React Native Developer (Kaunas / remote)

Requirements:
- 2+ years of experience with React or React Native
- Good knowledge of TypeScript
- Experience publishing mobile apps (iOS / Android)
- Working with REST APIs

Nice to have:
- Firebase
- CI/CD for mobile apps
- Agile experience
"""

my_response = post_match(MY_CV, MY_JOB_AD)
show_report({"id": "my_cv", "scenario": "Frontend developer → React Native role", "y": {"match_score": "?", "verdict": "?"}}, my_response)

## 11. Save the results

All answers are saved to `outputs/results.json` (example id, expected `y`, predicted `y_pred` and `meta`). In Colab, download the file from the 📁 Files panel.

In [ ]:
#@title 💾 11.1 Save results
out_dir = Path("outputs")
out_dir.mkdir(exist_ok=True)
out_path = out_dir / "results.json"
out_path.write_text(json.dumps([
    {"id": run["example"]["id"], "y": run["example"]["y"], "y_pred": run["response"]["result"], "meta": run["response"]["meta"]}
    for run in runs
], indent=2, ensure_ascii=False), encoding="utf-8")
print(f"✅ Saved {len(runs)} results to {out_path.resolve()}")

## 12. Conclusions

- The **CVMatch AI** prototype works end-to-end: from the input (CV + job ad) to a validated JSON answer, a chart and a report card. Every step matches a UML component from the README.
- **Gemini** does what simple rules cannot: it understands context ("Redis may mean experience with queues"), recognises skills written in different ways and writes concrete, personalised recommendations.
- The **Result Validator** makes the system more reliable: the score always stays within 0–100 and the verdict always matches the score. Without an API key the keyword fallback still returns an answer in the same format, but it is much less accurate.
- **Limitations:** 4 examples are not enough for a statistically reliable evaluation, and the expected `y` scores are expert estimates, so they are partly subjective.
- **Ethics:** a CV contains personal data. A real system would need GDPR compliance (consent, no storage of CVs, anonymisation before sending text to an external API), and the final decision about a candidate must be made by a human.